Upload Stage 3 / Original Dataset

In [1]:
import pandas as pd
import numpy as np
from google.colab import files

# Upload Stage 3 extracted dataset
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

Saving railway_delay_analyzer_finaldataset to railway_delay_analyzer_finaldataset
Dataset loaded successfully!
Dataset Shape: (50000, 40)


Check Dataset Information

In [2]:
print("Dataset Information:")
print("=" * 50)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Dataset Information:
Rows: 50000
Columns: 40

Column Names:
['date', 'train_no', 'train_name', 'type_code', 'station_no', 'station_name', 'station_full_name', 'station_zone', 'station_address', 'distance_from_origin', 'arrival_day', 'arrival_time', 'departure_day', 'departure_time', 'delay', 'year', 'month', 'day_of_week', 'scheduled_arrival_minutes', 'scheduled_departure_minutes', 'delay_status_from_source', 'delay_over_15_source_units', 'public_holiday', 'festival_or_holiday', 'is_weekend', 'distance_from_origin_complete_km', 'route_total_distance_km', 'train_route', 'segment_distance_km', 'scheduled_arrival_time_complete', 'scheduled_arrival_datetime', 'actual_arrival_datetime', 'arrival_delay_minutes', 'scheduled_vs_actual_arrival_status', 'weather_condition', 'temperature_c', 'rainfall_mm', 'congestion_index', 'congestion_level', 'operational_disruption']

Data Types:
date                                   object
train_no                                int64
train_name            

Missing-Value Percentage

In [3]:
missing_count = df.isnull().sum()

missing_percentage = (missing_count / len(df)) * 100

missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": missing_count.values,
    "Missing_Percentage": missing_percentage.values
})

missing_report = missing_report.sort_values(
    by="Missing_Percentage",
    ascending=False
)

print("Missing Value Report:")
print(missing_report.to_string(index=False))

Missing Value Report:
                            Column  Missing_Count  Missing_Percentage
         scheduled_arrival_minutes          17817              35.634
                      arrival_time          17817              35.634
                    departure_time          16631              33.262
       scheduled_departure_minutes          16631              33.262
              distance_from_origin          15249              30.498
                       arrival_day          15249              30.498
                     departure_day          15249              30.498
                              date              0               0.000
                      station_zone              0               0.000
                 station_full_name              0               0.000
                      station_name              0               0.000
                        station_no              0               0.000
                         type_code              0               0.00

Duplicate Check

In [4]:
duplicate_count = df.duplicated().sum()

print("Duplicate Records:", duplicate_count)

if duplicate_count == 0:
    print("No duplicate records found.")
else:
    print("Duplicate records found.")

Duplicate Records: 0
No duplicate records found.


Data Type Correction

In [5]:


df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

df["scheduled_arrival_datetime"] = pd.to_datetime(
    df["scheduled_arrival_datetime"],
    errors="coerce"
)

df["actual_arrival_datetime"] = pd.to_datetime(
    df["actual_arrival_datetime"],
    errors="coerce"
)

# Numeric columns
numeric_columns = [
    "train_no",
    "station_no",
    "distance_from_origin",
    "arrival_day",
    "departure_day",
    "delay",
    "year",
    "month",
    "scheduled_arrival_minutes",
    "scheduled_departure_minutes",
    "arrival_delay_minutes",
    "temperature_c",
    "rainfall_mm",
    "congestion_index"
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

print("Data types corrected successfully!")

print("\nCorrected Date Types:")
print(df[[
    "date",
    "scheduled_arrival_datetime",
    "actual_arrival_datetime"
]].dtypes)

Data types corrected successfully!

Corrected Date Types:
date                          datetime64[ns]
scheduled_arrival_datetime    datetime64[ns]
actual_arrival_datetime       datetime64[ns]
dtype: object




Train & Station Identifier Cleaning

In [6]:
df["train_no"] = df["train_no"].astype("string").str.strip()
df["station_no"] = df["station_no"].astype("string").str.strip()

# Clean train and station names
df["train_name"] = (
    df["train_name"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df["station_name"] = (
    df["station_name"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df["station_full_name"] = (
    df["station_full_name"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print("Train and station identifiers standardized successfully!")

print("\nSample Train Identifiers:")
print(df["train_no"].head().tolist())

print("\nSample Station Identifiers:")
print(df["station_no"].head().tolist())

Train and station identifiers standardized successfully!

Sample Train Identifiers:
['961', '961', '961', '961', '961']

Sample Station Identifiers:
['1', '1', '1', '1', '1']


Missing Values Treatment

In [7]:
numeric_imputation_columns = [
    "distance_from_origin",
    "arrival_day",
    "departure_day",
    "scheduled_arrival_minutes",
    "scheduled_departure_minutes"
]

for column in numeric_imputation_columns:
    median_value = df[column].median()
    df[column] = df[column].fillna(median_value)

# Time-related columns
time_columns = [
    "arrival_time",
    "departure_time"
]

for column in time_columns:
    df[column] = df[column].fillna("UNKNOWN")

print("Missing values treated successfully!")

Missing values treated successfully!


Verify Missing Values After Cleaning

In [8]:
remaining_missing = df.isnull().sum()

remaining_missing = remaining_missing[
    remaining_missing > 0
]

print("Remaining Missing Values:")

if len(remaining_missing) == 0:
    print("No missing values remaining.")
else:
    print(remaining_missing)

Remaining Missing Values:
No missing values remaining.


Outlier Detection using IQR

In [9]:
outlier_columns = [
    "delay",
    "arrival_delay_minutes",
    "temperature_c",
    "rainfall_mm",
    "congestion_index"
]

outlier_report = []

for column in outlier_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = (
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    )

    outlier_count = outliers.sum()

    outlier_report.append({
        "Column": column,
        "Q1": Q1,
        "Q3": Q3,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Outlier_Count": outlier_count
    })

outlier_report = pd.DataFrame(outlier_report)

print("Outlier Detection Report:")
print(outlier_report.to_string(index=False))

Outlier Detection Report:
               Column     Q1        Q3  Lower_Bound  Upper_Bound  Outlier_Count
                delay 12.000 112.00000  -138.000000   262.000000           4447
arrival_delay_minutes 12.000 112.00000  -138.000000   262.000000           4447
        temperature_c 22.300  30.10000    10.600000    41.800000             15
          rainfall_mm  0.000   0.00000     0.000000     0.000000           8986
     congestion_index  0.418   0.69725    -0.000875     1.116125              0


Outlier Treatment

In [10]:
for column in outlier_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[column] = df[column].clip(
        lower=lower_bound,
        upper=upper_bound
    )

print("Outliers treated successfully using IQR clipping.")

Outliers treated successfully using IQR clipping.


Remove Duplicate Records After Cleaning

In [11]:
duplicates_before = df.duplicated().sum()

df = df.drop_duplicates()

duplicates_after = df.duplicated().sum()

print("Duplicate check after cleaning:")
print("Duplicates before removal:", duplicates_before)
print("Duplicates after removal:", duplicates_after)

Duplicate check after cleaning:
Duplicates before removal: 0
Duplicates after removal: 0


Before vs After Cleaning Comparison

In [12]:
original_rows = 50000

# Current cleaned dataset information
cleaned_rows = len(df)

original_missing = 0

# Missing values from actual original dataset
original_missing = (
    15249 +
    15249 +
    17817 +
    15249 +
    16631 +
    16631 +
    17817 +
    16631
)

cleaned_missing = df.isnull().sum().sum()

comparison = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Total Missing Values",
        "Duplicate Rows"
    ],
    "Before Cleaning": [
        original_rows,
        35,
        original_missing,
        0
    ],
    "After Cleaning": [
        cleaned_rows,
        df.shape[1],
        cleaned_missing,
        df.duplicated().sum()
    ]
})

print("BEFORE vs AFTER CLEANING")
print("=" * 60)
print(comparison.to_string(index=False))

BEFORE vs AFTER CLEANING
              Metric  Before Cleaning  After Cleaning
                Rows            50000           50000
             Columns               35              40
Total Missing Values           131274               0
      Duplicate Rows                0               0


Final Validation

In [13]:
print("FINAL DATA VALIDATION")
print("=" * 50)

print("Final Shape:", df.shape)
print("Total Missing Values:", df.isnull().sum().sum())
print("Total Duplicate Rows:", df.duplicated().sum())

print("\nFinal Data Types:")
print(df.dtypes)

print("\nFirst 5 Cleaned Records:")
display(df.head())

FINAL DATA VALIDATION
Final Shape: (50000, 40)
Total Missing Values: 0
Total Duplicate Rows: 0

Final Data Types:
date                                  datetime64[ns]
train_no                              string[python]
train_name                            string[python]
type_code                                     object
station_no                            string[python]
station_name                          string[python]
station_full_name                     string[python]
station_zone                                  object
station_address                               object
distance_from_origin                         float64
arrival_day                                  float64
arrival_time                                  object
departure_day                                float64
departure_time                                object
delay                                        float64
year                                           int64
month                                 

,date,train_no,train_name,type_code,station_no,station_name,station_full_name,station_zone,station_address,distance_from_origin,...,scheduled_arrival_datetime,actual_arrival_datetime,arrival_delay_minutes,scheduled_vs_actual_arrival_status,weather_condition,temperature_c,rainfall_mm,congestion_index,congestion_level,operational_disruption
0,2025-02-12,961,VALLEY QUEEN SPL,EXP-TRAINS,1,MJ,MARWAR JN,NWR,"Pali, Rajasthan, India",0.0,...,2025-02-12 09:37:00,2025-02-12 09:37:00,0.0,Early/on-time,Cloudy,16.3,0.0,0.27,Low,Normal operations
1,2025-02-13,961,VALLEY QUEEN SPL,EXP-TRAINS,1,MJ,MARWAR JN,NWR,"Pali, Rajasthan, India",0.0,...,2025-02-13 09:37:00,2025-02-13 09:37:00,0.0,Early/on-time,Clear,15.8,0.0,0.27,Low,Normal operations
2,2025-02-22,961,VALLEY QUEEN SPL,EXP-TRAINS,1,MJ,MARWAR JN,NWR,"Pali, Rajasthan, India",0.0,...,2025-02-22 09:37:00,2025-02-22 09:37:00,0.0,Early/on-time,Cool & Clear,16.6,0.0,0.37,Moderate,Normal operations
3,2025-02-23,961,VALLEY QUEEN SPL,EXP-TRAINS,1,MJ,MARWAR JN,NWR,"Pali, Rajasthan, India",0.0,...,2025-02-23 09:37:00,2025-02-23 09:37:00,0.0,Early/on-time,Cloudy,21.8,0.0,0.37,Moderate,Normal operations
4,2025-02-24,961,VALLEY QUEEN SPL,EXP-TRAINS,1,MJ,MARWAR JN,NWR,"Pali, Rajasthan, India",0.0,...,2025-02-24 09:37:00,2025-02-24 09:37:00,0.0,Early/on-time,Cloudy,19.7,0.0,0.27,Low,Normal operations


Save Cleaned Dataset

In [14]:
df.to_csv(
    "stage4_cleaned_data.csv",
    index=False
)

# Save reports
missing_report.to_csv(
    "stage4_missing_value_report.csv",
    index=False
)

outlier_report.to_csv(
    "stage4_outlier_report.csv",
    index=False
)

comparison.to_csv(
    "stage4_before_after_comparison.csv",
    index=False
)

print("Stage 4 files saved successfully!")

print("\nCreated Files:")
print("1. stage4_cleaned_data.csv")
print("2. stage4_missing_value_report.csv")
print("3. stage4_outlier_report.csv")
print("4. stage4_before_after_comparison.csv")

Stage 4 files saved successfully!

Created Files:
1. stage4_cleaned_data.csv
2. stage4_missing_value_report.csv
3. stage4_outlier_report.csv
4. stage4_before_after_comparison.csv
